Question:
 
Best Selling Item 
 
Find the best-selling item for each month (no need to separate months by year). The best-selling item is determined by the highest total sales amount, calculated as: total_paid = unitprice * quantity. A negative quantity indicates a return or cancellation (the invoice number begins with 'C'. To calculate sales, ignore returns and cancellations. Output the month, description of the item, and the total amount paid. 
 

In [0]:
%sql
CREATE TABLE online_retail (
    invoiceno STRING,
    stockcode STRING,
    description STRING,
    quantity INT,
    invoicedate DATE,
    unitprice DOUBLE,
    customerid INT,
    country STRING
)

In [0]:
%sql
INSERT INTO online_retail VALUES
-- January
('540001','10001','RED MUG',10,DATE '2011-01-05',2.50,101,'United Kingdom'),
('540002','10002','BLUE MUG',5,DATE '2011-01-10',3.00,102,'United Kingdom'),
('C540003','10001','RED MUG',-2,DATE '2011-01-12',2.50,101,'United Kingdom'),

-- February
('540004','10001','RED MUG',8,DATE '2011-02-03',2.50,103,'United Kingdom'),
('540005','10003','GREEN PLATE',15,DATE '2011-02-07',4.00,104,'France'),
('C540006','10003','GREEN PLATE',-3,DATE '2011-02-08',4.00,104,'France'),

-- March
('540007','10002','BLUE MUG',20,DATE '2011-03-01',3.00,105,'Germany'),
('540008','10003','GREEN PLATE',5,DATE '2011-03-05',4.00,106,'Germany'),
('540009','10004','YELLOW CUP',25,DATE '2011-03-10',1.50,107,'Spain'),

-- April
('540010','10001','RED MUG',30,DATE '2011-04-02',2.50,108,'Italy'),
('540011','10004','YELLOW CUP',10,DATE '2011-04-06',1.50,109,'Italy'),

-- May
('540012','10005','WHITE BOWL',40,DATE '2011-05-01',2.00,110,'Netherlands'),
('540013','10001','RED MUG',10,DATE '2011-05-03',2.50,111,'Netherlands'),

-- June
('540014','10002','BLUE MUG',50,DATE '2011-06-01',3.00,112,'Belgium'),
('C540015','10002','BLUE MUG',-5,DATE '2011-06-02',3.00,112,'Belgium'),

-- July
('540016','10003','GREEN PLATE',60,DATE '2011-07-01',4.00,113,'UK'),
('540017','10005','WHITE BOWL',20,DATE '2011-07-05',2.00,114,'UK'),

-- August
('540018','10004','YELLOW CUP',70,DATE '2011-08-01',1.50,115,'France'),

-- September
('540019','10001','RED MUG',80,DATE '2011-09-01',2.50,116,'Germany'),

-- October
('540020','10002','BLUE MUG',90,DATE '2011-10-01',3.00,117,'Spain'),

-- November
('540021','10003','GREEN PLATE',100,DATE '2011-11-01',4.00,118,'Italy'),

-- December
('540022','10005','WHITE BOWL',120,DATE '2011-12-01',2.00,119,'Netherlands');

num_affected_rows,num_inserted_rows
22,22


In [0]:
%sql

Select * from online_retail;

invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country
540001,10001,RED MUG,10,2011-01-05,2.5,101,United Kingdom
540002,10002,BLUE MUG,5,2011-01-10,3.0,102,United Kingdom
C540003,10001,RED MUG,-2,2011-01-12,2.5,101,United Kingdom
540004,10001,RED MUG,8,2011-02-03,2.5,103,United Kingdom
540005,10003,GREEN PLATE,15,2011-02-07,4.0,104,France
C540006,10003,GREEN PLATE,-3,2011-02-08,4.0,104,France
540007,10002,BLUE MUG,20,2011-03-01,3.0,105,Germany
540008,10003,GREEN PLATE,5,2011-03-05,4.0,106,Germany
540009,10004,YELLOW CUP,25,2011-03-10,1.5,107,Spain
540010,10001,RED MUG,30,2011-04-02,2.5,108,Italy


In [0]:
%sql
WITH sales AS (
    SELECT 
        MONTH(invoicedate) AS month,
        description,
        SUM(unitprice * quantity) AS total_paid
    FROM online_retail
    WHERE quantity > 0
      AND invoiceno NOT LIKE 'C%'
    GROUP BY MONTH(invoicedate), description
),
ranked AS (
    SELECT 
        month,
        description,
        total_paid,
        ROW_NUMBER() OVER (
            PARTITION BY month 
            ORDER BY total_paid DESC
        ) AS rnk
    FROM sales
)
SELECT 
    month,
    description,
    total_paid
FROM ranked
WHERE rnk = 1
ORDER BY month;

month,description,total_paid
1,RED MUG,25.0
2,GREEN PLATE,60.0
3,BLUE MUG,60.0
4,RED MUG,75.0
5,WHITE BOWL,80.0
6,BLUE MUG,150.0
7,GREEN PLATE,240.0
8,YELLOW CUP,105.0
9,RED MUG,200.0
10,BLUE MUG,270.0
